# Exploratory Data Analysis: Mental Health & Twitter Graph

This notebook performs the initial exploratory data analysis on the mental health text and Twitter network datasets. The goal is to understand the distribution of emotions, text properties, and the structural characteristics of the social network graph before advancing to modeling.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, count, desc

# Initialize Spark Session
spark = SparkSession.builder.appName("EDA_MentalHealth").getOrCreate()

# Load datasets
mental_health_df = spark.read.csv("mental_health.csv", header=True, inferSchema=True)
twitter_graph_df = spark.read.csv("twitter_graph.csv", header=True, inferSchema=True)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [3]:
pip install pyspark

  DEPRECATION: Building 'pyspark' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyspark'. Discussion can be found at https://github.com/pypa/pip/issues/6334



     ---------------------------------------- 0.0/455.9 MB ? eta -:--:--
     - ----------------------------------- 19.4/455.9 MB 105.1 MB/s eta 0:00:05
     --- --------------------------------- 43.0/455.9 MB 111.3 MB/s eta 0:00:04
     ----- ------------------------------- 68.2/455.9 MB 113.3 MB/s eta 0:00:04
     ----- -------------------------------- 71.3/455.9 MB 88.6 MB/s eta 0:00:05
     -------- ----------------------------- 96.5/455.9 MB 94.4 MB/s eta 0:00:04
     --------- --------------------------- 120.3/455.9 MB 98.1 MB/s eta 0:00:04
     ----------- ------------------------ 144.2/455.9 MB 100.8 MB/s eta 0:00:04
     ------------- ---------------------- 168.0/455.9 MB 102.8 MB/s eta 0:00:03
     -------------- --------------------- 188.7/455.9 MB 102.6 MB/s eta 0:00:03
     ---------------- ------------------- 212.3/455.9 MB 104.0 MB/s eta 0:00:03
     ------------------ ----------------- 235.9/455.9 MB 105.1 MB/s eta 0:00:03
     -------------------- --------------- 259.

### Emotional Category Distribution
Analyzing the class balance across the six psychological categories (Positive, Neutral, Anger, Depression, Sadness, and Anxiety) to identify potential class imbalances before training machine learning models.

In [ ]:
emotion_distribution = mental_health_df.groupBy("Emotion").count().orderBy(col("count").desc())
emotion_distribution.show()

### Text Length Variability
Assessing the length of the text entries to understand the distribution of message sizes, which will inform our tokenization, padding strategies, and feature engineering for NLP.

In [ ]:
mental_health_df = mental_health_df.withColumn("text_length", length(col("Text")))
mental_health_df.select("text_length").summary("count", "mean", "min", "max").show()

### Twitter Graph Network Structure
Evaluating the scale of the directed relationships and checking for missing or inconsistent account metadata. This provides baseline sizing for the subsequent GraphFrames implementation.

In [ ]:
print(f"Total Network Edges: {twitter_graph_df.count()}")

twitter_graph_df.select("Source Followers", "Destination Followers").summary("count", "mean", "max").show()
twitter_graph_df.groupBy("Source Verified").count().show()